In [12]:
from pathlib import Path

import numpy as np
import torch
from librosa.effects import pitch_shift
from sklearn.model_selection import train_test_split
from torch import nn
from IPython.display import Audio
from torchaudio.functional import add_noise
from torchvision import transforms
import torchaudio.functional as F

from CustomSpeachDataset import CustomSpeachDataset
from SpeachClassifierModel import SpeachClassifierModel
from augmentations import audio_transform, RandomSpeedPitchShift, AddRandomNoise

In [13]:
device = device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [14]:
MODELS_DIR = Path("checkpoints")
MODELS_DIR.mkdir(exist_ok=True)

In [15]:
dataset = CustomSpeachDataset(Path("preprocessed_dataset"), preload=True)
dataset.to(device)

Preloading dataset from disk...


In [16]:
train_val_indices, test_indices = train_test_split(
    range(len(dataset)),
    test_size=0.2,
    stratify=dataset.y.cpu(),
    random_state=42
)

class_weights = 1.0 / dataset.label_counts
all_sample_weights = np.array([class_weights[y] for y in dataset.y.cpu()])

y_train = dataset.y.cpu()[train_val_indices]
train_val_dataset = torch.utils.data.Subset(dataset, train_val_indices)
train_val_sample_weights = all_sample_weights[train_val_indices]
len(train_val_dataset), len(y_train)

(14679, 14679)

In [55]:
add_noise = AddRandomNoise(min_snr=0.001, max_snr=0.008).to(device)
pitch_shift = RandomSpeedPitchShift(sample_rate=8000, min_steps=-3, max_steps=3).to(device)

In [83]:
samples, y = train_val_dataset[np.arange(32).tolist()]
samples = add_noise(samples)
samples.shape

torch.Size([32, 1, 8000])

In [19]:
samples.device

device(type='cuda', index=0)

In [20]:
factor = 2.0 ** (3 / 12.0)
factor

1.189207115002721

In [21]:
resampled = F.resample(samples, 8000, 15000).cpu()
resampled.shape

torch.Size([3, 1, 15000])

In [22]:
Audio(F.resample(samples, 8000, 15000).cpu(), rate=8000)

ValueError: Array audio input must be a 1D or 2D array